# CWE Forest API Demo

This notebook demonstrates all available APIs for the `cwe_tree` module, which provides programmatic access to the Common Weakness Enumeration (CWE) forest structure.

The CWE forest is a hierarchical graph of security weaknesses where:
- **Nodes** represent individual CWE weaknesses
- **Edges** represent parent-child relationships
- **Multiple roots** create an independent tree structure (forest)

## Table of Contents
1. [Installation & Import](#installation--import)
2. [Basic Node Access](#basic-node-access)
3. [Parent-Child Navigation](#parent-child-navigation)
4. [Forest Structure](#forest-structure)
5. [Graph Traversal](#graph-traversal)
6. [Metadata Retrieval](#metadata-retrieval)
7. [Visualization](#visualization)
8. [Advanced Examples](#advanced-examples)

## Installation & Import

In [ ]:
# Import the module
# !pip install cwe-tree
from cwe_tree import query

# `query` is a pre-loaded singleton CweForest instance
print(f"CweForest type: {type(query)}")
print(f"CweForest instance: {query}")

## Basic Node Access

### `get_node(cwe_id) -> Optional[CweNode]`

Retrieve a specific CWE node by its ID. The ID is automatically normalized (e.g., "284" becomes "CWE-284").

In [ ]:
# Get a node with full ID
node_79 = query.get_node("CWE-79")
if node_79:
    print(f"Found: {node_79.cwe_id}")
    print(f"Name: {node_79.name}")
    print(f"Abstract: {node_79.abstract}")
else:
    print("Node not found")

In [ ]:
# Get a node with normalized ID (ID normalization works automatically)
node_284 = query.get_node("284")  # Automatically becomes "CWE-284"
if node_284:
    print(f"Found: {node_284.cwe_id}")
    print(f"Name: {node_284.name}")
else:
    print("Node not found")

In [ ]:
# Handle non-existent nodes
non_existent = query.get_node("CWE-99999")
print(f"Result for non-existent node: {non_existent}")

if non_existent is None:
    print("Node not found - gracefully handled")

## Parent-Child Navigation

### `get_parents(cwe_id) -> Set[CweNode]`

Retrieve all parent nodes of a given CWE weakness.

In [ ]:
# Get parents of a node
node = query.get_node("CWE-284")

parents = query.get_parents("CWE-284")
print(f"Parents of CWE-284:")
for parent in parents:
    print(f"  - {parent.cwe_id}: {parent.name}")

print(f"\nTotal parents: {len(parents)}")

### `get_children(cwe_id) -> Set[CweNode]`

Retrieve all child nodes of a given CWE weakness.

In [ ]:
# Get children of a node
children = query.get_children("CWE-1")
print(f"Children of CWE-1:")
for child in list(children)[:5]:  # Show first 5
    print(f"  - {child.cwe_id}: {child.name}")

print(f"\nTotal children: {len(children)}")

## Forest Structure

### `get_root_nodes() -> list[CweNode]`

Retrieve all root nodes in the CWE forest. Root nodes are nodes with no parents, forming the top level of independent tree hierarchies.

In [ ]:
# Get all root nodes
roots = query.get_root_nodes()
print(f"Forest has {len(roots)} root node(s):\n")

for root in roots:
    print(f"Root: {root.cwe_id}")
    print(f"  Name: {root.name}")
    print(f"  Abstract: {root.abstract}")
    print()

## Graph Traversal

These methods are provided by the underlying `AbcGraphQuerier` base class from cpg2py.

### `succ(node) -> Iterable[CweNode]`

Returns all successor nodes (children) connected via outgoing edges.

In [ ]:
# Use succ() directly
node = query.get_node("CWE-1")

successors = list(query.succ(node))[:5]  # Get first 5
print(f"First 5 successors of {node.cwe_id}:")
for succ in successors:
    print(f"  - {succ.cwe_id}: {succ.name}")

### `prev(node) -> Iterable[CweNode]`

Returns all predecessor nodes (parents) connected via incoming edges.

In [ ]:
# Use prev() directly
node = query.get_node("CWE-284")

predecessors = list(query.prev(node))
print(f"Predecessors of {node.cwe_id}:")
for pred in predecessors:
    print(f"  - {pred.cwe_id}: {pred.name}")

### `descendants(node, max_depth=None) -> Iterable[CweNode]`

Performs breadth-first traversal to find all nodes reachable from the source node (all descendants).

In [ ]:
# Get all descendants
root = query.get_root_nodes()[0]
all_descendants = list(query.descendants(root))

print(f"Total descendants of {root.cwe_id}: {len(all_descendants)}")
print(f"First 10 descendants:")
for desc in all_descendants[:10]:
    print(f"  - {desc.cwe_id}")

In [ ]:
# Get descendants with depth limit
root = query.get_root_nodes()[0]
descendants_depth_2 = list(query.descendants(root, max_depth=2))

print(f"Descendants of {root.cwe_id} up to depth 2: {len(descendants_depth_2)}")

### `ancestors(node, max_depth=None) -> Iterable[CweNode]`

Performs breadth-first traversal to find all nodes from which the source node is reachable (all ancestors).

In [ ]:
# Get all ancestors
node = query.get_node("CWE-284")
all_ancestors = list(query.ancestors(node))

print(f"All ancestors of {node.cwe_id}: {len(all_ancestors)}")
for anc in all_ancestors:
    print(f"  - {anc.cwe_id}: {anc.name}")

### `nodes(predicate=None) -> Iterable[CweNode]`

Iterate over all nodes in the forest, optionally filtered by a predicate function.

In [ ]:
# Get all nodes
all_nodes = list(query.nodes())
print(f"Total nodes in forest: {len(all_nodes)}")

In [ ]:
# Filter nodes by predicate
class_nodes = list(query.nodes(lambda n: n.abstract == "Class"))
print(f"Total 'Class' abstract type nodes: {len(class_nodes)}")
print(f"First 5 Class nodes:")
for node in class_nodes[:5]:
    print(f"  - {node.cwe_id}: {node.name}")

### `edges(predicate=None) -> Iterable[CweEdge]`

Iterate over all edges in the forest, optionally filtered by a predicate function.

In [ ]:
# Get all edges
all_edges = list(query.edges())
print(f"Total edges in forest: {len(all_edges)}")

# Show first 5 edges
print(f"\nFirst 5 edges:")
for edge in all_edges[:5]:
    print(f"  {edge.fid} -> {edge.tid} ({edge.eid})")

### `first_node(predicate=None) -> Optional[CweNode]`

Returns the first node matching the predicate, or None if no match found.

In [ ]:
# Find first node with specific property
first_base = query.first_node(lambda n: n.abstract == "Base")
if first_base:
    print(f"First Base node found: {first_base.cwe_id}")
    print(f"  Name: {first_base.name}")

## Metadata Retrieval

### `get_metadata(cwe_id) -> Optional[dict]`

Retrieve complete metadata for a node including its relationships.

In [ ]:
# Get comprehensive metadata
import json

metadata = query.get_metadata("CWE-79")
print(json.dumps(metadata, indent=2))

### `get_layer(cwe_id) -> dict`

Retrieve layer information showing the depth of a node in different root hierarchies.

In [ ]:
# Get layer information
layer = query.get_layer("CWE-284")
print(f"Layer information for CWE-284:")
print(json.dumps(layer, indent=2))

## Visualization

### `show(cwe_id=None) -> None`

Display the forest structure with tree-like ASCII formatting.

In [ ]:
# Display entire forest from root nodes
print("Displaying entire CWE forest structure:")
print("="*60)
query.show()

In [ ]:
# Display specific subtree
print("Displaying subtree from CWE-1:")
print("="*60)
query.show("CWE-1")

In [ ]:
# Display specific subtree (with normalization)
print("Displaying subtree from node 284:")
print("="*60)
query.show("284")  # Automatically normalized to CWE-284

## Advanced Examples

Practical use cases combining multiple APIs.

### Example 1: Find all Variant-type weaknesses

In [ ]:
# Find all Variant type nodes
variant_nodes = list(query.nodes(lambda n: n.abstract == "Variant"))
print(f"Total Variant nodes: {len(variant_nodes)}")
print(f"\nFirst 10 Variant nodes:")
for node in variant_nodes[:10]:
    print(f"  {node.cwe_id}: {node.name}")

### Example 2: Analyze a specific weakness hierarchy

In [ ]:
# Analyze CWE-79 (XSS) hierarchy
cwe_79 = query.get_node("CWE-79")

print(f"=== CWE-79 (XSS) Analysis ===")
print(f"Name: {cwe_79.name}")
print(f"Abstract: {cwe_79.abstract}")

# Get parents
parents = query.get_parents("CWE-79")
print(f"\nParents ({len(parents)}):")
for parent in parents:
    print(f"  - {parent.cwe_id}: {parent.name}")

# Get children
children = query.get_children("CWE-79")
print(f"\nChildren ({len(children)}):")
for child in list(children)[:5]:
    print(f"  - {child.cwe_id}: {child.name}")
if len(children) > 5:
    print(f"  ... and {len(children) - 5} more")

# Get all descendants
descendants = list(query.descendants(cwe_79))
print(f"\nAll descendants: {len(descendants)}")

### Example 3: Find nodes by name pattern

In [ ]:
# Find all nodes with "Buffer" in the name
buffer_nodes = list(query.nodes(lambda n: "Buffer" in n.name))
print(f"Nodes with 'Buffer' in name: {len(buffer_nodes)}")
print()
for node in buffer_nodes:
    print(f"  {node.cwe_id}: {node.name}")

### Example 4: Trace ancestry path

In [ ]:
# Find complete ancestry path from a specific node to root
def get_path_to_root(forest, node_id):
    """Get the path from a node to its root ancestor."""
    path = []
    current = forest.get_node(node_id)
    visited = set()
    
    while current and current.cwe_id not in visited:
        path.append(current)
        visited.add(current.cwe_id)
        
        parents = forest.get_parents(current.cwe_id)
        if parents:
            current = list(parents)[0]  # Follow first parent
        else:
            break
    
    return path

# Example: trace path for CWE-284
path = get_path_to_root(query, "CWE-284")
print(f"Path from CWE-284 to root:")
for i, node in enumerate(path):
    indent = "  " * i
    print(f"{indent}-> {node.cwe_id}: {node.name}")

### Example 5: Forest Statistics

In [ ]:
# Compute forest statistics
all_nodes = list(query.nodes())
all_edges = list(query.edges())
roots = query.get_root_nodes()

# Count by abstract type
abstract_counts = {}
for node in all_nodes:
    abstract = node.abstract
    abstract_counts[abstract] = abstract_counts.get(abstract, 0) + 1

print("=== CWE Forest Statistics ===")
print(f"Total nodes: {len(all_nodes)}")
print(f"Total edges: {len(all_edges)}")
print(f"Root nodes: {len(roots)}")

print(f"\nNodes by abstract type:")
for abstract, count in sorted(abstract_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {abstract}: {count}")

## Summary

The `cwe_tree` module provides a comprehensive API for:

1. **Node Access:** Get individual nodes by ID with automatic normalization
2. **Relationship Navigation:** Find parents, children, ancestors, and descendants
3. **Graph Traversal:** Use cpg2py's underlying traversal methods (succ, prev, descendants, ancestors)
4. **Forest Exploration:** Identify root nodes and overall structure
5. **Metadata Retrieval:** Access comprehensive node information and layer data
6. **Visualization:** Display tree structures with ASCII formatting

### Key Design Principles

- **Type Safety:** All methods return properly typed CweNode/CweEdge objects
- **ID Normalization:** CWE IDs are automatically normalized (e.g., "284" → "CWE-284")
- **Lazy Evaluation:** Traversal methods return iterables for memory efficiency
- **Error Handling:** Non-existent nodes return None gracefully
- **Forest Support:** Handles multiple independent root trees

For more details, see:
- `docs/design.md` - Architectural design and concepts
- `docs/traversal.md` - Detailed traversal API reference